# Notebook 03 — Analysis Dashboard
### Plotly Charts · Creator Leaderboard · Trend Analysis

Interactive visualizations for the Q3/Q4 2024 VALORANT creator campaign.

**Run after** `02_measurement_framework.ipynb` — reads from `data/modeled/`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

DATA_MODELED = '../data/modeled'
DATA_RAW     = '../data/raw'

# VALORANT brand colors
VALORANT_RED   = '#FF4655'
VALORANT_DARK  = '#0F1923'
ACCENT_BLUE    = '#00D4FF'
ACCENT_YELLOW  = '#FFE033'

CREATOR_COLORS = {
    'TenZ':           VALORANT_RED,
    'tarik':          '#00D4FF',
    'Kyedae':         '#FF9500',
    'aceu':           '#7B68EE',
    'Sinatraa':       '#00FF88',
    'Protatomonster': '#FFE033',
}

master  = pd.read_csv(f'{DATA_MODELED}/creator_campaign_metrics.csv')
inc     = pd.read_csv(f'{DATA_MODELED}/incrementality_q3_q4.csv')
alerts  = pd.read_csv(f'{DATA_MODELED}/flagging_alerts.csv')
trends  = pd.read_csv(f'{DATA_RAW}/search_interest_monthly.csv')

print(f'Master: {len(master)} rows | Alerts: {len(alerts)} | Trends: {len(trends)}')

## Chart 1 — Q4 Creator Leaderboard (ER %)

In [ ]:
q4 = master[master['quarter']=='Q4'].groupby('creator').agg(
    avg_er   = ('er',          'mean'),
    avg_cpe  = ('cpe',         'mean'),
    total_v  = ('total_views', 'sum'),
).reset_index().sort_values('avg_er', ascending=True)

colors = [CREATOR_COLORS.get(c, '#888') for c in q4['creator']]

fig1 = go.Figure(go.Bar(
    y         = q4['creator'],
    x         = q4['avg_er'],
    orientation = 'h',
    marker_color = colors,
    text      = q4['avg_er'].round(2).astype(str) + '%',
    textposition = 'outside',
))
fig1.add_vline(x=3.87, line_dash='dash', line_color='white',
               annotation_text='Benchmark 3.87%', annotation_font_color='white')
fig1.update_layout(
    title      = 'Q4 2024 Creator Leaderboard — Avg Engagement Rate',
    xaxis_title = 'Engagement Rate (%)',
    plot_bgcolor = VALORANT_DARK,
    paper_bgcolor = VALORANT_DARK,
    font_color = 'white',
    height     = 400,
)
fig1.show()

## Chart 2 — Monthly ER Trends (All Creators)

In [ ]:
fig2 = go.Figure()

for creator in master['creator'].unique():
    sub = master[master['creator']==creator].sort_values('month')
    fig2.add_trace(go.Scatter(
        x    = sub['month'],
        y    = sub['er'],
        name = creator,
        mode = 'lines+markers',
        line = dict(color=CREATOR_COLORS.get(creator,'#888'), width=2),
        marker = dict(size=8),
    ))

fig2.add_hline(y=3.87, line_dash='dash', line_color='white',
               annotation_text='Benchmark 3.87%')
fig2.update_layout(
    title  = 'Monthly Engagement Rate — Q3 + Q4 2024',
    yaxis_title = 'ER (%)',
    xaxis_title = 'Month',
    plot_bgcolor = VALORANT_DARK,
    paper_bgcolor = VALORANT_DARK,
    font_color = 'white',
    height = 500,
)
fig2.show()

## Chart 3 — ER vs CPE Scatter (Q4, bubble = total views)

In [ ]:
q4_s = master[master['quarter']=='Q4'].groupby('creator').agg(
    avg_er  = ('er',          'mean'),
    avg_cpe = ('cpe',         'mean'),
    total_v = ('total_views', 'sum'),
).reset_index()

fig3 = go.Figure()
for _, row in q4_s.iterrows():
    fig3.add_trace(go.Scatter(
        x    = [row['avg_cpe']],
        y    = [row['avg_er']],
        mode = 'markers+text',
        name = row['creator'],
        text = [row['creator']],
        textposition = 'top center',
        marker = dict(
            size  = np.sqrt(row['total_v']) / 100,
            color = CREATOR_COLORS.get(row['creator'], '#888'),
            opacity = 0.85,
        ),
    ))

fig3.add_hline(y=3.87,  line_dash='dot', line_color='gray')
fig3.add_vline(x=0.25,  line_dash='dot', line_color='gray')

fig3.update_layout(
    title   = 'Q4 Efficiency Map — ER vs CPE (bubble = total views)',
    xaxis_title = 'Avg CPE ($) — lower is better',
    yaxis_title = 'Avg ER (%) — higher is better',
    plot_bgcolor  = VALORANT_DARK,
    paper_bgcolor = VALORANT_DARK,
    font_color    = 'white',
    height = 500,
    showlegend = False,
)
fig3.show()

## Chart 4 — Google Trends + Total Views Overlay

In [ ]:
monthly_views = master.groupby('month')['total_views'].sum().reset_index()
trend_data    = trends.copy()
combined      = monthly_views.merge(trend_data, on='month', how='left')

fig4 = make_subplots(specs=[[{'secondary_y': True}]])

fig4.add_trace(go.Bar(
    x    = combined['month'],
    y    = combined['total_views'],
    name = 'Total Program Views',
    marker_color = VALORANT_RED,
    opacity = 0.8,
), secondary_y=False)

fig4.add_trace(go.Scatter(
    x    = combined['month'],
    y    = combined['avg_interest'],
    name = 'VALORANT Search Interest',
    mode = 'lines+markers',
    line = dict(color=ACCENT_YELLOW, width=3),
), secondary_y=True)

fig4.update_layout(
    title = 'Program Views vs. VALORANT Search Interest (Jul–Dec 2024)',
    plot_bgcolor  = VALORANT_DARK,
    paper_bgcolor = VALORANT_DARK,
    font_color    = 'white',
    height = 450,
)
fig4.update_yaxes(title_text='Total Views', secondary_y=False)
fig4.update_yaxes(title_text='Google Trends (0–100)', secondary_y=True)
fig4.show()

## Chart 5 — Incrementality: Q3 → Q4 ER Delta

In [ ]:
inc_sorted = inc.sort_values('delta_er_pp', ascending=False)
bar_colors = [VALORANT_RED if d < 0 else '#00D4FF' for d in inc_sorted['delta_er_pp']]

fig5 = go.Figure(go.Bar(
    x    = inc_sorted['creator'],
    y    = inc_sorted['delta_er_pp'],
    marker_color = bar_colors,
    text = inc_sorted['delta_er_pp'].apply(lambda x: f'{x:+.2f}pp'),
    textposition = 'outside',
))
fig5.add_hline(y=0, line_color='white', line_width=1)
fig5.update_layout(
    title   = 'Incrementality: Q4 ER vs Q3 ER (pp change)',
    yaxis_title = 'ΔER (percentage points)',
    plot_bgcolor  = VALORANT_DARK,
    paper_bgcolor = VALORANT_DARK,
    font_color    = 'white',
    height = 400,
)
fig5.show()

## Alert Summary Table

In [ ]:
print('=== AUTOMATED ALERTS SUMMARY ===')
print(alerts['alert_type'].value_counts())
print()
print(alerts.to_string(index=False))

## Q1 2025 Recommendations

In [ ]:
recs = pd.DataFrame([
    {'Creator': 'tarik',           'Action': 'SCALE +50%',       'Rationale': 'Highest Q4 ER (4.45%), only improving creator, CPE acceptable'},
    {'Creator': 'TenZ',            'Action': 'RENEW',            'Rationale': 'Audience loyalty, viral upside, CPE acceptable'},
    {'Creator': 'Kyedae',          'Action': 'RENEW',            'Rationale': 'Above benchmark Q4, audience loyalty despite declining MoM trend'},
    {'Creator': 'Sinatraa',        'Action': 'MONITOR',          'Rationale': 'Low ER, watch another full cycle before budget decision'},
    {'Creator': 'Protatomonster',  'Action': 'ROTATE OUT',       'Rationale': 'Flat ER vs cost, CPE review-tier 3 consecutive months'},
    {'Creator': 'aceu',            'Action': 'RENEGOTIATE -30%', 'Rationale': 'CPE $3.46 — 14x above benchmark, consistent under-performance'},
])
print(recs.to_string(index=False))